
# Telco Customer Churn

Este notebook demonstra **fim a fim**, em um único arquivo, como:
1) **Preprocessar** o dataset (limpeza, tipagem, imputação, anti-leakage)  
2) **Treinar** um modelo (scikit-learn `Pipeline` com `OneHotEncoder` + `RandomForestClassifier`)  
3) **Registrar/Aprovar** (simulação de *guardrails* com *thresholds* configuráveis)



## 0. Paths e Configuração

In [14]:

import os
import json
import pandas as pd
import math

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn import __version__ as skl_version
from packaging.version import Version
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)
from joblib import dump

# === Ajuste estes caminhos conforme o seu ambiente ===
RAW_CSV      = os.getenv("RAW_CSV", "telcoChurn_input.csv")   # arquivo bruto
CLEAN_CSV    = os.getenv("CLEAN_CSV", "outputs/telcoChurn_clean.csv")     # arquivo limpo (saida do preprocess)
MODEL_PATH   = os.getenv("MODEL_PATH", "outputs/telcoChurn.joblib")  # modelo treinado
METRICS_JSON = os.getenv("METRICS_JSON", "outputs/telcoChurn_metrics.json")  # métricas salvas

# Thresholds do "register" (podem ser sobrescritos por env vars)
ACC_THRESH        = float(os.getenv("ACC_THRESH", "0.80"))
F1_THRESH         = float(os.getenv("F1_THRESH",  "0.80"))
PER_CLASS_F1_MIN  = float(os.getenv("PER_CLASS_F1_MIN", "0.75"))
AUC_MIN           = float(os.getenv("AUC_MIN", "0.80"))



## 1. Preprocess (limpeza e preparo)

Objetivos desta etapa:
- **Padronizar strings** (remover espaços / tipagem consistente)
- **Remover IDs** e colunas irrelevantes
- **Normalizar o alvo** (`Churn` → 0/1)
- **Converter numéricas textuais** (`TotalCharges`)
- **Imputar faltantes** (mediana em numéricas; `"Unknown"` em categóricas)
- **Remover colunas sem variação**
- **Anti-leakage**: remover quaisquer colunas relacionadas a *churn* (exceto o target)


In [15]:

# 1) Ler o CSV bruto
if not os.path.exists(RAW_CSV):
    raise FileNotFoundError(f"Não encontrei {RAW_CSV}. Ajuste a variável RAW_CSV no topo.")
df = pd.read_csv(RAW_CSV)
print("Shape bruto:", df.shape)
display(df.head(3))

# 2) Padronizar strings (sem transformar NaN em 'nan')
for c in df.select_dtypes(include=["object"]):
    df[c] = df[c].astype("string").str.strip()

# 3) Remover IDs e colunas irrelevantes
drop_cols = [c for c in df.columns if c.lower() in ["customerid", "id"]]
df = df.drop(columns=drop_cols, errors="ignore")

# 4) Alvo binário (0/1)
if "Churn" not in df.columns:
    raise ValueError("Coluna 'Churn' não encontrada no dataset de entrada.")
df["Churn"] = df["Churn"].astype(str).str.lower().map({"yes":1, "no":0})

# 5) Converter TotalCharges para numérico (strings vazias -> NaN)
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 6) Imputação simples
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(df[c].median())
    else:
        df[c] = df[c].fillna("Unknown")

# 7) Remover colunas sem variação (1 valor único)
nun = df.nunique()
low_var = nun[nun <= 1].index.tolist()
df = df.drop(columns=low_var, errors="ignore")

# 8) Anti-leakage: remove qualquer coluna (exceto 'Churn') contendo 'churn' no nome
leaky = [c for c in df.columns if ("churn" in c.lower()) and (c != "Churn")]
df = df.drop(columns=leaky, errors="ignore")

# 9) Salvar dataset limpo
df.to_csv(CLEAN_CSV, index=False)
print("✅ Preprocess concluído. Limpo:", df.shape, "-> salvo em", CLEAN_CSV)


Shape bruto: (844, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,1248.000000,Female,1,Yes,No,54.0,Yes,Yes,DSL,No,...,No,Yes,No,No,One year,Yes,Electronic check,46.640000,2493.62000,Yes
1,1308.000000,Male,0,Yes,Yes,14.0,Yes,Yes,Fiber optic,Yes,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),63.240000,881.18000,No
2,1586.587472,Male,1,No,No,14.0,Yes,No,Fiber optic,No,...,Yes,No,Yes,No,Month-to-month,No,Electronic check,69.147237,1053.86259,Yes


✅ Preprocess concluído. Limpo: (844, 20) -> salvo em outputs/telcoChurn_clean.csv



## 2. Train (treino + métricas)

Decisões guiadas pela EDA:
- **OneHotEncoder** para categóricas, **handle_unknown='ignore'** (evita quebra na inferência se surgir nova categoria)
- **class_weight='balanced'** para mitigar desbalanceamento do alvo
- Métricas reportadas: **Accuracy**, **F1-macro**, **AUC**, **per-class metrics** e **confusion matrix**


In [16]:


# --- (A) Helper: OneHotEncoder compatível ---
def make_ohe():
    if Version(skl_version) >= Version("1.2"):
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# 1) Carregar dataset limpo
df_clean = pd.read_csv(CLEAN_CSV)
print("Shape limpo:", df_clean.shape)

# 2) Definir X e y
y = df_clean["Churn"].astype(int)
X = df_clean.drop(columns=["Churn"])

# 3) Separar colunas categóricas e numéricas
cat_cols = [c for c in X.columns if (X[c].dtype == "object" or str(X[c].dtype) == "string")]
num_cols = [c for c in X.columns if c not in cat_cols]

# 4) Pré-processamento
cat_pipe = make_pipeline(make_ohe())
num_pipe = make_pipeline(StandardScaler())

pre = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, num_cols),
    ],
    remainder="drop",
)

# 4.1) Classificador base
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    n_jobs=-1,
    random_state=42,
)

# 4.2) Montar pipeline
pipe = Pipeline(steps=[
    ("prep", pre),
    ("rf", rf),
])

# 5) Split estratificado holdout para avaliação final
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)


# Predições e probabilidades
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]


acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    y_test, y_pred, zero_division=0
)
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)

print(f"✅ Treinado! ACC={acc:.4f} | F1macro={f1_macro:.4f} | AUC={auc:.4f}")
print("Matriz de confusão:\n", cm)

# 8) Salvar métricas + modelo
metrics_payload = {
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "auc": float(auc),
    "per_class": [
        {"label": str(lbl), "precision": float(p), "recall": float(r), "f1": float(f1v), "support": int(s)}
        for lbl, p, r, f1v, s in zip(sorted(y.unique()), precision, recall, f1_per_class, support)
    ],
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "n_features": {"categorical": len(cat_cols), "numerical": len(num_cols)},
}

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

dump(pipe, MODEL_PATH, compress=3)
print("✅ Modelo salvo em", MODEL_PATH)
print("✅ Métricas salvas em", METRICS_JSON)


Shape limpo: (844, 20)
✅ Treinado! ACC=0.8166 | F1macro=0.8156 | AUC=0.9190
Matriz de confusão:
 [[63 22]
 [ 9 75]]
✅ Modelo salvo em outputs/telcoChurn.joblib
✅ Métricas salvas em outputs/telcoChurn_metrics.json



## 3. Register (simulação de *guardrails*)

A promoção do modelo é decidida por **thresholds** configuráveis (via variáveis de ambiente ou constantes no topo):
- `ACC_THRESH` (padrão 0.80)
- `F1_THRESH` (padrão 0.80)
- `PER_CLASS_F1_MIN` (padrão 0.75)
- `AUC_MIN` (padrão 0.80)

> **Ideia:** na sua plataforma, estes valores podem ser inputs do node `register` para aprovar/reprovar ao vivo.


In [17]:

# 1) Carregar métricas salvas
with open(METRICS_JSON, "r", encoding="utf-8") as f:
    metrics = json.load(f)

acc = metrics.get("accuracy")
f1_macro = metrics.get("f1_macro")
auc = metrics.get("auc")

# 2) Calcular F1 mínimo entre as classes
per_class = metrics.get("per_class") or []
f1_vals = []
for row in per_class:
    try:
        v = float(row.get("f1"))
    except (TypeError, ValueError):
        v = None
    if v is not None and not math.isnan(v):
        f1_vals.append(v)

min_f1 = min(f1_vals) if f1_vals else None

# 3) Decisão
reasons = []
passed = True

if f1_macro is None or f1_macro < F1_THRESH:
    passed = False; reasons.append(f"f1_macro {f1_macro:.3f} < {F1_THRESH:.3f}")
if acc is not None and acc < ACC_THRESH:
    passed = False; reasons.append(f"accuracy {acc:.3f} < {ACC_THRESH:.3f}")
if auc is None or auc < AUC_MIN:
    passed = False; reasons.append(f"auc {auc:.3f} < {AUC_MIN:.3f}")
if (min_f1 is not None) and (min_f1 < PER_CLASS_F1_MIN):
    passed = False; reasons.append(f"min per-class F1 {min_f1:.3f} < {PER_CLASS_F1_MIN:.3f}")

print("📊 Métricas:")
print(f"  ACC     : {acc:.4f}")
print(f"  F1-macro: {f1_macro:.4f}")
print(f"  AUC     : {auc:.4f}")
if min_f1 is not None:
    print(f"  Min F1 por classe: {min_f1:.4f}")

print("\nDecisão:", "✅ APROVADO" if passed else "❌ REPROVADO")
if reasons:
    print("Motivo(s):", "; ".join(reasons))
else:
    print("Motivo(s): atende a todos os thresholds.")


📊 Métricas:
  ACC     : 0.8166
  F1-macro: 0.8156
  AUC     : 0.9190
  Min F1 por classe: 0.8025

Decisão: ✅ APROVADO
Motivo(s): atende a todos os thresholds.
